# Wan 2.1 on a T4: which settings give a usable picture?

The AWS test clips looked bad. Our settings are below Wan's own defaults (steps 20 or 30 against 50,
flow_shift 3.0 against 5.0) and the Wan README recommends guidance 6 and flow_shift 8 to 12 for the 1.3B
model. This notebook runs nine variants of one 3 s clip (832x480, same seed, same prompt) and saves each
as an **mp4** in `/kaggle/working` (Output tab) so you can watch them side by side.

**Run 1 first, and compare it with the clip from AWS** (the `high` clip saved in `outputs/`): the AWS clip had
good anatomy but blown-out colour (flat cyan sea, orange cat clipped, white spray burned), while the earlier Kaggle
frames (a cat in a garden) looked natural. Run 1 uses the surfing prompt and the AWS settings. If run 1 is blown out too,
the prompt or settings cause it ("vivid colors" and "bright tropical sun" were in the prompt); if run 1 looks natural,
the cause is the AWS setup (torch 2.8.0 on an A10G against torch 2.11 on a T4).

Run the cells **one at a time with Shift+Enter, top to bottom**. Each run is its own cell; if one fails, go on.
A 30-step run takes about 15 minutes and a 50-step run about 25 (T4). Runs 2 and 3 tell you first which single
knob matters most; runs 5 and 8 can be skipped if time is short. Free, nothing on AWS.

Same settings as before: Accelerator **GPU T4 x2**, Internet **On**.


In [ ]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"   # one T4, like the real instance (must be set before torch loads)
import json, shutil, subprocess, sys, time

REPORT = []
def check(name, status, detail=""):
    """status: PASS, FAIL or INFO"""
    REPORT.append({"check": name, "status": status, "detail": detail})
    print(f"[{status}] {name}: {detail}")

import psutil, torch
gpu = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none"
vram = torch.cuda.get_device_properties(0).total_memory / 2**30 if torch.cuda.is_available() else 0
check("GPU is a T4", "PASS" if "T4" in gpu else "FAIL", f"{gpu}, {vram:.1f} GiB VRAM")
check("RAM", "INFO", f"{psutil.virtual_memory().total / 2**30:.1f} GiB total (g4dn.2xlarge has 32)")
free_tmp = shutil.disk_usage("/tmp").free / 2**30
check("Disk for weights in /tmp", "PASS" if free_tmp > 40 else "FAIL", f"{free_tmp:.0f} GiB free, need about 35")
check("torch", "INFO", torch.__version__ + " (cuda " + str(torch.version.cuda) + "); the image will carry torch 2.8.0")


In [ ]:
PINS = ['diffusers==0.35.2', 'transformers==4.57.6', 'accelerate==1.15.0', 'safetensors==0.8.0', 'sentencepiece==0.2.2', 'ftfy==6.3.1', 'regex==2026.9.29', 'imageio==2.38.1', 'imageio-ffmpeg==0.6.0', 'huggingface-hub==0.36.2']
r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", *PINS], capture_output=True, text=True)
print(r.stdout[-2000:], r.stderr[-2000:])
import importlib.metadata as md
got = {p: md.version(p) for p in ("diffusers", "transformers", "accelerate", "safetensors", "sentencepiece", "ftfy", "imageio", "imageio-ffmpeg")}
check("pinned packages installed", "PASS" if r.returncode == 0 else "FAIL", json.dumps(got))


In [ ]:
# The worker's own files, embedded by scripts/make_spike_notebook.py
import pathlib
FILES = {'__init__.py': '"""The GPU worker: takes jobs off the queue, runs the model, uploads the clip (spec section 4)."""\n', 'errors.py': '"""The ways a job attempt can end other than success."""\n\n\nclass Aborted(Exception):\n    """The generator stopped early because `should_abort()` turned true."""\n\n\nclass BadMessage(Exception):\n    """A queue message that cannot be tied to a job (no usable job_id or sub). Nobody to tell."""\n\n\nclass PermanentJobError(Exception):\n    """The job can never succeed. Becomes a `.failed` marker the user sees, so `message` must be\n    safe to show: no paths, no stack traces."""\n\n    def __init__(self, code, message, field=None):\n        super().__init__(message)\n        self.code = code\n        self.message = message\n        self.field = field\n\n    def to_json(self):\n        out = {"code": self.code}\n        if self.field:\n            out["field"] = self.field\n        out["message"] = self.message\n        return out\n', 'wan.py': '"""Wan 2.1 T2V 1.3B through diffusers\' WanPipeline (spec section 4, "Model").\n\nRun on a Kaggle T4 on 2026-11-20 (docs/wan_spike_findings.md): loads, clean frames in fp16 with an\nfp32 VAE, abort frees the GPU. Not yet run on a g4dn instance. Guidance and flow_shift are the\ndocumented defaults, not tuned.\n\ntorch and diffusers are imported inside methods, so the rest of the worker (and its tests)\nload without them.\n"""\nimport gc\nimport logging\nimport pathlib\nimport random\n\nfrom .errors import Aborted\n\nlog = logging.getLogger(__name__)\n\nFPS = 16\n# The manifest\'s `quality` choices: (width, height, steps). Kaggle T4 spike, 2026-11-20, 3 s clip:\n# fast about 4.7 min, high about 15 min (docs/wan_spike_findings.md). Sizes must be multiples of 16.\nQUALITY = {"fast": (640, 368, 20), "high": (832, 480, 30)}\nGUIDANCE_SCALE = 5.0\nFLOW_SHIFT = 3.0          # diffusers\' Wan docs: 3.0 for 480p, 5.0 for 720p\nFRAMES = {"3": 49, "5": 81}   # 16 fps, and the model needs 4n+1 frames\nMAX_SEED = 2147483647\nLOG_EVERY_STEPS = 5      # one progress line per this many denoising steps, plus the last\n\n\ndef num_frames(duration_seconds):\n    return FRAMES[duration_seconds]\n\n\ndef preset(quality):\n    return QUALITY[quality]\n\n\ndef pick_seed(params, rng=random):\n    seed = params.get("seed")\n    return seed if seed is not None else rng.randint(0, MAX_SEED)\n\n\nclass WanGenerator:\n    def __init__(self, models_dir, model_name, steps=None, guidance=None, flow_shift=None,\n                 default_negative_prompt=None):\n        self.path = str(pathlib.Path(models_dir) / model_name)\n        # None for each: the module default (or the quality preset) decides. Only the spike\n        # notebooks set these, to compare settings (plan 05); the worker uses the defaults.\n        self.steps = steps\n        self.guidance = guidance\n        self.flow_shift = flow_shift\n        self.default_negative_prompt = default_negative_prompt   # used when the job gives none\n        self.pipe = None\n\n    def load(self):\n        """Load the weights once, at start. The T4 has no bf16: transformer and text encoder in\n        fp16, VAE in fp32 (spec section 4). Components move to the GPU only while they run."""\n        import torch\n        from diffusers import AutoencoderKLWan, UniPCMultistepScheduler, WanPipeline\n\n        vae = AutoencoderKLWan.from_pretrained(self.path, subfolder="vae", torch_dtype=torch.float32)\n        pipe = WanPipeline.from_pretrained(self.path, vae=vae, torch_dtype=torch.float16)\n        flow_shift = FLOW_SHIFT if self.flow_shift is None else self.flow_shift\n        pipe.scheduler = UniPCMultistepScheduler.from_config(pipe.scheduler.config, flow_shift=flow_shift)\n        pipe.enable_model_cpu_offload()\n        self.pipe = pipe\n\n    def generate(self, job, inputs, should_abort, out_dir):\n        error = None\n        try:\n            return self._generate(job, should_abort, out_dir)\n        except Aborted as e:\n            # The traceback holds the pipeline\'s tensors, so drop it before freeing memory.\n            error = e.with_traceback(None)\n        finally:\n            if error is None:\n                self._free()\n        self._free()\n        raise error\n\n    def _free(self):\n        """An abort skips diffusers\' own cleanup: the transformer stayed on the GPU and the next\n        job ran out of memory (Kaggle spike). Put the model back on the CPU and drop the cache."""\n        import torch\n\n        self.pipe.maybe_free_model_hooks()\n        gc.collect()\n        torch.cuda.empty_cache()\n\n    def _generate(self, job, should_abort, out_dir):\n        import torch\n        from diffusers.utils import export_to_video\n\n        if should_abort():\n            raise Aborted()\n\n        seed = pick_seed(job.params)\n        width, height, preset_steps = preset(job.params["quality"])\n        total_steps = self.steps or preset_steps\n        frame_count = num_frames(job.params["duration_seconds"])\n        # Settings only, never the prompt (it may be private).\n        log.info("generating %dx%d, %d frames, %d steps", width, height, frame_count, total_steps)\n\n        def on_step(pipe, step, timestep, callback_kwargs):\n            if should_abort():\n                raise Aborted()          # leaves the denoising loop between two steps\n            done = step + 1\n            if done % LOG_EVERY_STEPS == 0 or done == total_steps:\n                log.info("step %d/%d", done, total_steps)\n            return callback_kwargs\n\n        frames = self.pipe(\n            prompt=job.prompt,\n            negative_prompt=job.params.get("negative_prompt") or self.default_negative_prompt or None,\n            width=width, height=height,\n            num_frames=frame_count,\n            num_inference_steps=total_steps,\n            guidance_scale=GUIDANCE_SCALE if self.guidance is None else self.guidance,\n            generator=torch.Generator("cpu").manual_seed(seed),\n            callback_on_step_end=on_step,\n        ).frames[0]\n        out = pathlib.Path(out_dir) / "clip.mp4"\n        export_to_video(frames, str(out), fps=FPS)\n        return out\n'}
root = pathlib.Path("/kaggle/working/spike")
(root / "worker").mkdir(parents=True, exist_ok=True)
for name, text in FILES.items():
    (root / "worker" / name).write_text(text)
sys.path.insert(0, str(root))
from worker.wan import WanGenerator, num_frames
from worker.errors import Aborted
print(sorted(p.name for p in (root / "worker").iterdir()))


In [ ]:
from huggingface_hub import snapshot_download
WEIGHTS = "/tmp/wan-weights/wan2-1-t2v-1-3b"      # same layout as /data/models/<model> on the instance
t = time.time()
snapshot_download("Wan-AI/Wan2.1-T2V-1.3B-Diffusers", revision="0fad780a534b6463e45facd96134c9f345acfa5b", local_dir=WEIGHTS,
                  allow_patterns=["model_index.json", "scheduler/*", "text_encoder/*", "tokenizer/*", "transformer/*", "vae/*"])
size = sum(f.stat().st_size for f in pathlib.Path(WEIGHTS).rglob("*") if f.is_file()) / 2**30
check("weights downloaded", "PASS" if 25 < size < 40 else "FAIL", f"{size:.1f} GiB in {time.time() - t:.0f} s (spec says about 29)")
os.environ["HF_HUB_OFFLINE"] = "1"                 # the image never reaches out to Hugging Face
os.environ["TRANSFORMERS_OFFLINE"] = "1"


In [ ]:
gen = WanGenerator("/tmp/wan-weights", "wan2-1-t2v-1-3b", steps=30)
t = time.time()
try:
    gen.load()
    check("model loads (fp16 transformer and text encoder, fp32 VAE, cpu offload)", "PASS",
          f"{time.time() - t:.0f} s, RAM now {psutil.Process().memory_info().rss / 2**30:.1f} GiB")
except Exception as e:
    check("model loads", "FAIL", repr(e)[:400])
    raise


In [ ]:
import types
PROMPT_A = "A cat walking through a sunlit garden, cinematic, detailed fur"
PROMPT_B = "A red balloon floating up above a city street at sunset, soft light"

import numpy as np, imageio
from PIL import Image
from IPython.display import display

def job(prompt, seconds, seed):
    return types.SimpleNamespace(prompt=prompt, params={"seed": seed, "duration_seconds": seconds, "quality": "high", "negative_prompt": ""})

def read_frames(path):
    rd = imageio.get_reader(str(path), "ffmpeg")
    frames = [np.asarray(f) for f in rd]
    rd.close()
    return frames

def contact_sheet(frames, name):
    idx = np.linspace(0, len(frames) - 1, 6).astype(int)
    thumbs = [Image.fromarray(frames[i]).resize((416, 240)) for i in idx]
    sheet = Image.new("RGB", (416 * 3, 240 * 2))
    for n, im in enumerate(thumbs):
        sheet.paste(im, ((n % 3) * 416, (n // 3) * 240))
    sheet.save(f"/kaggle/working/{name}.png")
    display(sheet)

def quality(frames):
    a = np.stack(frames).astype(np.float32)
    motion = float(np.abs(np.diff(a, axis=0)).mean())
    return {"mean": round(float(a.mean()), 1), "std": round(float(a.std()), 1), "motion": round(motion, 2),
            "black_frames": int(sum(f.max() < 8 for f in frames))}

def run(label, prompt, seconds, steps, seed, should_abort=lambda: False, show=True):
    """One generation. Returns (frames, seconds, peak_vram_gib) or raises."""
    gen.steps = steps
    out = pathlib.Path(f"/tmp/out-{label}"); out.mkdir(exist_ok=True)
    torch.cuda.reset_peak_memory_stats()
    t = time.time()
    path = gen.generate(job(prompt, seconds, seed), {}, should_abort, out)
    secs = time.time() - t
    peak = torch.cuda.max_memory_allocated() / 2**30
    frames = read_frames(path)
    if show:
        contact_sheet(frames, label)
    return frames, secs, peak

def judge(label, frames, secs, peak, want_frames, limit_s=900):
    q = quality(frames)
    clean = q["black_frames"] == 0 and q["std"] > 10 and q["mean"] > 8 and not np.isnan(q["mean"])
    moving = q["motion"] > 0.5
    ok = len(frames) == want_frames and clean and moving and secs <= limit_s
    why = f"{len(frames)} frames (want {want_frames}), {secs:.0f} s, peak VRAM {peak:.1f} GiB (contact sheet attached), quality {q}"
    if not clean: why += " | BLACK/FLAT FRAMES: try fp32 transformer (spec section 5)"
    if not moving: why += " | FRAMES BARELY MOVE"
    if secs > limit_s: why += f" | over {limit_s} s"
    check(label, "PASS" if ok else "FAIL", why)


In [ ]:
import shutil
from diffusers import UniPCMultistepScheduler

PROMPT_SURF = ("A fluffy orange tabby cat surfing a glassy turquoise wave on a small white surfboard, paws spread for "
               "balance, sea spray flying, bright tropical sun, cinematic slow motion, side-tracking camera, sharp focus, vivid colors")
PROMPT_SIMPLE = "An orange cat riding a surfboard on a calm blue sea, sunny day, clear sky"
# What the site sent on AWS (the owner's short negative prompt) and Wan's own default, from the diffusers WanPipeline docstring.
NEG_OWNER = "blurry, distorted, extra limbs, deformed paws, text, watermark, low quality, jittery"
NEG_OFFICIAL = ("Bright tones, overexposed, static, blurred details, subtitles, style, works, paintings, images, static, overall gray, "
                "worst quality, low quality, JPEG compression residue, ugly, incomplete, extra fingers, poorly drawn hands, "
                "poorly drawn faces, deformed, disfigured, misshapen limbs, fused fingers, still picture, messy background, "
                "three legs, many people in the background, walking backwards")
SEED = 1234
RESULTS = []

def set_shift(shift):
    """flow_shift is read when the scheduler is built, so rebuild it instead of reloading the model (about 2 min)."""
    gen.pipe.scheduler = UniPCMultistepScheduler.from_config(gen.pipe.scheduler.config, flow_shift=shift)

def grid_run(label, steps, guidance, shift, negative, prompt=PROMPT_SURF, seed=SEED):
    """One 3 s clip at 832x480 with the given settings. Saves /kaggle/working/<label>.mp4 and a contact sheet."""
    gen.steps, gen.guidance = steps, guidance
    set_shift(shift)
    j = types.SimpleNamespace(prompt=prompt, params={"seed": seed, "duration_seconds": "3", "quality": "high",
                                                     "negative_prompt": negative})
    out = pathlib.Path(f"/tmp/out-{label}"); out.mkdir(exist_ok=True)
    torch.cuda.reset_peak_memory_stats()
    t = time.time()
    path = gen.generate(j, {}, lambda: False, out)
    secs = time.time() - t
    shutil.copy(path, f"/kaggle/working/{label}.mp4")
    frames = read_frames(path)
    contact_sheet(frames, label)
    q = quality(frames)
    RESULTS.append({"run": label, "steps": steps, "guidance": guidance, "flow_shift": shift,
                    "negative": "official" if negative == NEG_OFFICIAL else ("owner" if negative == NEG_OWNER else "none"),
                    "seconds": round(secs), "mean": q["mean"], "std": q["std"], "motion": q["motion"]})
    print(RESULTS[-1], "- watch /kaggle/working/" + label + ".mp4")


## 1. Baseline (what ran on AWS: bad)

In [ ]:
grid_run("run1-baseline", steps=30, guidance=5.0, shift=3.0, negative=NEG_OWNER)

## 2. flow_shift alone (README range 8 to 12)

In [ ]:
grid_run("run2-shift8", steps=30, guidance=5.0, shift=8.0, negative=NEG_OWNER)

## 3. Steps alone (Wan and diffusers default is 50)

In [ ]:
grid_run("run3-steps50", steps=50, guidance=5.0, shift=3.0, negative=NEG_OWNER)

## 4. The README recipe: 50 steps, guidance 6, shift 8

In [ ]:
grid_run("run4-readme-recipe", steps=50, guidance=6.0, shift=8.0, negative=NEG_OWNER)

## 5. Top of the README shift range (optional)

In [ ]:
grid_run("run5-shift12", steps=50, guidance=6.0, shift=12.0, negative=NEG_OWNER)

## 6. Wan's official negative prompt on the baseline

In [ ]:
grid_run("run6-official-negative", steps=30, guidance=5.0, shift=3.0, negative=NEG_OFFICIAL)

## Pick the best settings

In [ ]:
# Look at runs 1 to 6 first, then set BEST to the settings you liked most (default: the README recipe).
BEST = dict(steps=50, guidance=6.0, shift=8.0)
print("BEST =", BEST)


## 7. Best settings plus Wan's official negative prompt

In [ ]:
grid_run("run7-best-plus-negative", negative=NEG_OFFICIAL, **BEST)

## 8. A simpler prompt (optional)

In [ ]:
grid_run("run8-simple-prompt", negative=NEG_OFFICIAL, prompt=PROMPT_SIMPLE, **BEST)

## 9. Second seed on the best settings (is it luck?)

In [ ]:
grid_run("run9-second-seed", negative=NEG_OFFICIAL, seed=777, **BEST)

## Report

In [ ]:
# REPORT: copy everything this cell prints and paste it back, then download the mp4 files from the Output tab.
print("run | steps | guidance | shift | negative | seconds | mean | std | motion")
for r in RESULTS:
    print(f"{r['run']} | {r['steps']} | {r['guidance']} | {r['flow_shift']} | {r['negative']} | {r['seconds']} | {r['mean']} | {r['std']} | {r['motion']}")
json.dump(RESULTS, open("/kaggle/working/grid_report.json", "w"), indent=1)
print("\nAlso saved: grid_report.json. Watch the mp4 files; the numbers only catch black or frozen clips, not ugly ones.")
